In [1]:
# Histogram Calculation using CUDA in Python (Numba)

import numpy as np
from numba import cuda
import math
import time

# CUDA Kernel for Histogram
@cuda.jit
def histogram_kernel(data, histo):

    idx = cuda.grid(1)

    # Check bounds
    if idx < data.size:

        value = data[idx]

        # Atomic add to avoid race condition
        cuda.atomic.add(histo, value, 1)


# Main Function
def compute_histogram(data, bins):

    # Allocate histogram array
    histo = np.zeros(bins, dtype=np.int32)

    # Copy arrays to GPU
    d_data = cuda.to_device(data)
    d_histo = cuda.to_device(histo)

    # CUDA Configuration
    threads_per_block = 256
    blocks_per_grid = math.ceil(data.size / threads_per_block)

    # Launch Kernel
    histogram_kernel[blocks_per_grid, threads_per_block](d_data, d_histo)

    # Copy result back to CPU
    result = d_histo.copy_to_host()

    return result


# Driver Code
if __name__ == "__main__":

    # Random data between 0 and 9
    data = np.random.randint(0, 10, size=1000).astype(np.int32)

    bins = 10

    print("Input Data:")
    print(data)

    start = time.time()

    histogram = compute_histogram(data, bins)

    end = time.time()

    print("\nHistogram Result:")
    for i in range(bins):
        print(f"Bin {i}: {histogram[i]}")

    print("\nExecution Time:", end - start, "seconds")

Input Data:
[3 6 2 0 0 5 7 6 1 9 1 7 5 2 7 1 9 0 9 5 6 6 1 6 3 2 5 8 6 1 6 3 5 6 6 7 6
 1 8 2 7 0 9 2 0 4 4 0 7 0 9 5 9 8 5 4 6 0 7 2 9 6 4 9 1 3 2 4 0 9 3 0 2 1
 9 8 5 2 1 7 6 0 8 9 6 2 8 0 1 3 4 2 0 7 7 2 8 6 8 8 3 4 1 1 4 2 3 9 9 3 4
 6 3 3 7 4 9 1 4 9 3 2 1 2 7 0 2 8 9 2 2 6 0 2 4 8 9 4 7 5 2 2 5 3 2 9 5 2
 2 5 4 0 0 7 2 7 1 1 6 4 8 6 3 9 2 7 8 7 1 0 5 9 2 4 8 3 1 6 5 0 1 2 0 4 9
 6 1 5 0 4 0 9 9 9 5 2 9 3 1 4 3 4 1 0 8 0 9 5 9 2 9 3 2 0 2 4 0 7 2 6 3 2
 6 8 5 1 3 7 0 3 2 6 3 8 1 6 0 9 1 8 4 9 0 0 7 9 5 9 2 2 6 2 5 9 6 9 3 4 6
 5 8 1 8 5 5 3 0 8 2 9 3 5 8 4 6 9 0 3 5 9 0 6 4 8 3 2 6 8 2 9 7 6 5 3 9 1
 7 3 9 4 1 7 7 6 4 1 6 9 3 4 7 6 6 3 4 4 3 7 3 9 0 4 5 2 5 4 9 5 6 1 2 9 5
 0 2 1 6 6 6 4 0 3 8 2 3 7 3 7 3 8 8 3 5 7 9 4 4 0 3 7 6 0 3 5 5 1 3 6 1 5
 3 3 8 2 8 0 7 6 1 2 9 6 5 9 9 5 5 5 0 2 4 2 0 1 1 5 2 3 1 2 6 4 0 6 3 4 2
 4 5 8 4 3 4 8 0 2 6 3 7 6 4 0 2 7 2 9 4 6 4 1 4 4 1 2 3 9 3 8 6 6 5 6 1 3
 8 5 3 9 4 2 4 8 5 1 5 3 5 7 3 5 4 9 0 6 2 8 1 7 3 9 3 3 3 6 4 9 5 9 3 8 3
 4 5 6 7 8 9 

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))



Histogram Result:
Bin 0: 87
Bin 1: 93
Bin 2: 98
Bin 3: 104
Bin 4: 103
Bin 5: 116
Bin 6: 102
Bin 7: 94
Bin 8: 89
Bin 9: 114

Execution Time: 2.6657469272613525 seconds
